# STEP 04｜Ontology × AutoMLで次の四半期を予測

架空の8社について、過去92日程度の実績から次の92日のEBITDAを予測します。
現在のOntologyから実体化されたGraphへGQL（beta）を実行し、イベント → 影響額 →
日次指標と影響額 → 取引活動の関係を取得。Lakehouseの金額と照合し、
予測時点までの影響額・経路数をAutoMLの追加特徴量にします。

**3Qで学習、4Qを探索的に評価**し、4Qまでで再学習した次Qの予測も表示。
GraphなしAutoML・直近平均と比べます。将来のイベントや為替の実績は入力しません。
Graphは現在のスナップショットのため、発生日にデータが利用可能だったと仮定します。
1年の合成データ・重複する教師期間では、実世界への一般化や厳密な季節性は検証できません。

結果はこのNotebookの出力に保存します。GraphとLakehouseのデータは変更しません。
「すべて実行」はGraph読み取りとSpark計算を行い、Fabric容量を消費します。


In [ ]:
if "html" in globals():
    display(HTML(html))
else:
    print("保存済みの実行結果がない場合は「すべて実行」を選択してください。")


## 再実行用のパラメーターと処理


In [ ]:
EXPECTED_WORKSPACE = "__WORKSPACE_ID__"
ONTOLOGY_NAME = "TradingOntology_Multisector"
NOTEBOOK_NAME = "TradingDemo_04_Ontology_AutoML_予測"


In [ ]:
"""Time-aware 92-day EBITDA forecasts from graph-linked historical effects."""
from __future__ import annotations

import numpy as np
import pandas as pd

HORIZON = 92
HOLDOUT_ORIGIN = pd.Timestamp("2026-06-30")
FUTURE_ORIGIN = pd.Timestamp("2026-09-30")
VALIDATION_ORIGIN = pd.Timestamp("2026-03-30")
FIRST_ORIGIN = pd.Timestamp("2025-10-28")
GRAPH_FEATURES = ["graph_amount_28", "graph_amount_91", "graph_paths_28", "graph_paths_91"]


def prepare_features(metrics, effects, links, paths):
    """Verify graph/source paths, then construct features using only past rows."""
    m = metrics.loc[
        (metrics["metric_code"] == "EBITDA") & (metrics["scenario_tag"] == "actual"),
        ["metric_id", "metric_date", "entity_id", "amount_group"],
    ].copy()
    m["metric_date"] = pd.to_datetime(m["metric_date"]).dt.normalize()
    if m["metric_id"].duplicated().any() or m.duplicated(["entity_id", "metric_date"]).any():
        raise ValueError("Duplicate actual EBITDA metric")
    m["amount_oku"] = pd.to_numeric(m["amount_group"]) / 100_000_000
    if m["amount_oku"].isna().any():
        raise ValueError("Missing EBITDA amount")

    e = effects[["effect_id", "event_id", "entity_id", "effect_date", "delta_ebitda_full_jpy"]].copy()
    e["effect_date"] = pd.to_datetime(e["effect_date"]).dt.normalize()
    if e["effect_id"].duplicated().any() or links["effect_id"].duplicated().any():
        raise ValueError("Duplicate effect or effect-to-metric link")
    source = e.merge(links[["effect_id", "metric_id"]], on="effect_id", validate="one_to_one")
    source = source.merge(
        m[["metric_id", "entity_id", "metric_date"]],
        on="metric_id", suffixes=("", "_metric"), validate="many_to_one",
    )
    if len(source) != len(e):
        raise ValueError("Some effects have no actual EBITDA metric")
    if not ((source["entity_id"] == source["entity_id_metric"]) &
            (source["effect_date"] == source["metric_date"])).all():
        raise ValueError("Effect and metric entity/date do not match")

    p = paths[["effect_id", "event_id", "entity_id", "metric_id", "event_start_date"]].copy()
    p["event_start_date"] = pd.to_datetime(p["event_start_date"]).dt.normalize()
    if p["effect_id"].duplicated().any() or set(p["effect_id"]) != set(source["effect_id"]):
        raise ValueError("Graph and Lakehouse effect membership differ")
    verified = source.merge(
        p, on=["effect_id", "event_id", "entity_id", "metric_id"], validate="one_to_one",
    )
    if len(verified) != len(source) or not (verified["event_start_date"] <= verified["effect_date"]).all():
        raise ValueError("Graph path mapping or event dates differ from Lakehouse")
    verified["effect_oku"] = pd.to_numeric(verified["delta_ebitda_full_jpy"]) / 100_000_000
    daily_effects = verified.groupby(["entity_id", "effect_date"], as_index=False).agg(
        effect_oku=("effect_oku", "sum"), graph_paths=("effect_id", "size"),
    ).rename(columns={"effect_date": "metric_date"})
    m = m.merge(daily_effects, on=["entity_id", "metric_date"], how="left", validate="one_to_one")
    m[["effect_oku", "graph_paths"]] = m[["effect_oku", "graph_paths"]].fillna(0)

    companies = sorted(m["entity_id"].unique())
    if len(companies) != 8:
        raise ValueError(f"Expected eight subsidiaries, got {len(companies)}")
    frames = []
    for company, group in m.groupby("entity_id", sort=True):
        group = group.sort_values("metric_date").reset_index(drop=True)
        expected = pd.date_range("2025-10-01", "2026-09-30", freq="D")
        if not pd.DatetimeIndex(group["metric_date"]).equals(expected):
            raise ValueError(f"Missing, duplicate, or unexpected daily dates for {company}")
        values = group["amount_oku"].to_numpy(dtype=float)
        cumulative = np.r_[0.0, values.cumsum()]
        labels = np.full(len(group), np.nan)
        labels[:-HORIZON] = cumulative[HORIZON + 1:] - cumulative[1:-HORIZON]
        group["target_oku"] = labels
        group["target_end"] = group["metric_date"] + pd.Timedelta(days=HORIZON)
        for days in (7, 28, 91):
            group[f"mean_{days}"] = group["amount_oku"].rolling(days, min_periods=days).mean()
        group["std_28"] = group["amount_oku"].rolling(28, min_periods=28).std()
        # Early training examples have 28–90 days of history; expose the window length.
        group["mean_91"] = group["amount_oku"].rolling(91, min_periods=28).mean()
        group["history_91"] = np.minimum(np.arange(1, len(group) + 1), 91)
        for days in (28, 91):
            group[f"graph_amount_{days}"] = group["effect_oku"].rolling(days, min_periods=1).sum()
            group[f"graph_paths_{days}"] = group["graph_paths"].rolling(days, min_periods=1).sum()
        for name in companies:
            group[f"company_{name}"] = int(company == name)
        frames.append(group)
    frame = pd.concat(frames, ignore_index=True)
    base = ["mean_7", "mean_28", "mean_91", "std_28", "history_91"]
    base += [f"company_{name}" for name in companies]
    return frame, base, {
        "graph_paths": len(verified),
        "companies": len(companies),
        "first_date": str(m["metric_date"].min().date()),
        "last_date": str(m["metric_date"].max().date()),
    }


def rows_at(frame, origin):
    rows = frame.loc[frame["metric_date"] == origin].sort_values("entity_id").copy()
    if len(rows) != 8:
        raise ValueError(f"Expected eight rows at {origin.date()}")
    return rows


def training_rows(frame, known_through):
    eligible = frame.loc[
        (frame["metric_date"] >= FIRST_ORIGIN) &
        (frame["target_end"] <= known_through) &
        frame["target_oku"].notna()
    ].copy()
    eligible = eligible.loc[((eligible["metric_date"] - FIRST_ORIGIN).dt.days % 7) == 0]
    if eligible.empty or eligible["target_end"].max() > known_through:
        raise ValueError("No training labels available at the requested cutoff")
    return eligible.sort_values(["metric_date", "entity_id"])


def _xy(rows, columns):
    x = rows[columns].to_numpy(dtype=float)
    y = rows["target_oku"].to_numpy(dtype=float)
    if not np.isfinite(x).all() or not np.isfinite(y).all():
        raise ValueError("Training or validation data contains missing/nonfinite values")
    return x, y


def search_model(frame, columns, time_budget=20):
    from flaml import AutoML

    train = training_rows(frame, VALIDATION_ORIGIN)
    validation = rows_at(frame, VALIDATION_ORIGIN)
    if validation["target_end"].max() > HOLDOUT_ORIGIN:
        raise ValueError("Model selection used Q4 data")
    x_train, y_train = _xy(train, columns)
    x_val, y_val = _xy(validation, columns)
    model = AutoML()
    model.fit(
        X_train=x_train, y_train=y_train, X_val=x_val, y_val=y_val,
        task="regression", metric="mae", eval_method="holdout",
        estimator_list=["rf", "extra_tree"], time_budget=time_budget,
        n_jobs=1, seed=41, sample=False, retrain_full=False,
        mlflow_logging=False, log_file_name="", verbose=0,
    )
    if model.model is None or model.best_estimator is None:
        raise RuntimeError("AutoML did not produce a model")
    return model, {"training_rows": len(train), "validation_rows": len(validation),
                   "validation_mae_company_oku": float(model.best_loss)}


def refit_predict(search, frame, columns, origin):
    train = training_rows(frame, origin)
    x_train, y_train = _xy(train, columns)
    config = dict(search.best_config)
    config.pop("FLAML_sample_size", None)
    config.setdefault("n_jobs", 1)
    model = type(search.model)(task="regression", **config)
    model.fit(x_train, y_train)
    selected = rows_at(frame, origin)
    features = selected[columns].to_numpy(dtype=float)
    if not np.isfinite(features).all():
        raise ValueError("Forecast features contain missing/nonfinite values")
    prediction = np.asarray(model.predict(features), dtype=float)
    if prediction.shape != (8,) or not np.isfinite(prediction).all():
        raise ValueError("Model produced invalid subsidiary predictions")
    return prediction, len(train)


def run_comparison(frame, base_columns, time_budget=20):
    """Compare identical point-in-time inputs, then refit without Q4 tuning."""
    q4 = rows_at(frame, HOLDOUT_ORIGIN)
    actual = q4["target_oku"].to_numpy(dtype=float)
    if not np.isfinite(actual).all():
        raise ValueError("Held-out Q4 actuals are incomplete")
    future = rows_at(frame, FUTURE_ORIGIN)
    entity_ids = future["entity_id"].tolist()
    if entity_ids != q4["entity_id"].tolist() or len(set(entity_ids)) != len(entity_ids):
        raise ValueError("Holdout and forecast rows do not describe the same companies")
    scores = []
    summaries = []
    for name, columns in (
        ("AutoML", base_columns),
        ("AutoML + Graph", base_columns + GRAPH_FEATURES),
    ):
        search, validation = search_model(frame, columns, time_budget)
        q4_prediction, q4_training = refit_predict(search, frame, columns, HOLDOUT_ORIGIN)
        future_prediction, future_training = refit_predict(search, frame, columns, FUTURE_ORIGIN)
        scores.append((name, q4_prediction, future_prediction))
        summaries.append({
            "model": name, "best_estimator": search.best_estimator,
            "validation": validation, "q4_training_rows": q4_training,
            "future_training_rows": future_training,
        })
    scores.insert(0, (
        "直近91日日平均", q4["mean_91"].to_numpy(dtype=float) * HORIZON,
        future["mean_91"].to_numpy(dtype=float) * HORIZON,
    ))
    results = []
    for name, prediction, future_prediction in scores:
        total, actual_total = float(prediction.sum()), float(actual.sum())
        results.append({
            "model": name, "q4_forecast_oku": total, "q4_actual_oku": actual_total,
            "q4_error_oku": total - actual_total,
            "q4_error_percent": (total - actual_total) / actual_total * 100,
            "q4_company_mae_oku": float(np.abs(prediction - actual).mean()),
            "next_q_forecast_oku": float(future_prediction.sum()),
            # 画面の積み上げ棒（業種別）に使う。合計は next_q_forecast_oku と一致する
            "next_q_by_entity_oku": [
                {"entity_id": entity, "amount_oku": float(value)}
                for entity, value in zip(entity_ids, future_prediction)
            ],
        })
    return {"results": results, "models": summaries,
            "validation_origin": str(VALIDATION_ORIGIN.date()),
            "q4_origin": str(HOLDOUT_ORIGIN.date()),
            "future_origin": str(FUTURE_ORIGIN.date()),
            "horizon_days": HORIZON}


In [ ]:
"""Read live Ontology graph and Lakehouse data; save results only in this notebook."""
import base64
import datetime as dt
import json
import re
import time
import urllib.error
import urllib.parse
import urllib.request

import mlflow
import notebookutils
import pandas as pd
from IPython.display import HTML, display

mlflow.autolog(disable=True)
API = "https://api.fabric.microsoft.com/v1/"
context = notebookutils.runtime.context
workspace_id = context["currentWorkspaceId"]
notebook_id = context["currentNotebookId"]
if workspace_id != EXPECTED_WORKSPACE or context["currentNotebookName"] != NOTEBOOK_NAME:
    raise RuntimeError("Tradingの指定された予測Notebook以外では実行しません。")


def api(path, method="GET", body=None, skill="spark-cli"):
    url = path if path.startswith(API) else API + path
    if not url.startswith(API):
        raise ValueError("Unexpected Fabric API host")
    headers = {
        "Authorization": "Bearer " + notebookutils.credentials.getToken("pbi"),
        "Content-Type": "application/json",
        "Accept": "application/json",
        "x-ms-fabric-skill": skill,
    }
    request = urllib.request.Request(
        url, data=None if body is None else json.dumps(body).encode(),
        headers=headers, method=method,
    )
    try:
        with urllib.request.urlopen(request, timeout=120) as response:
            data = response.read()
            return (json.loads(data) if data else None,
                    {key.lower(): value for key, value in response.headers.items()})
    except urllib.error.HTTPError as error:
        raise RuntimeError(f"Fabric API HTTP {error.code}: 接続・権限・クエリを確認してください。") from error


def list_all(path):
    results = []
    for _ in range(20):
        page, _ = api(path)
        results.extend(page["value"])
        path = page.get("continuationUri")
        if not path:
            if page.get("continuationToken"):
                raise RuntimeError("Catalog pagination is incomplete")
            return results
    raise RuntimeError("Catalog pagination exceeded limit")


def one_named(values, name):
    matches = [item for item in values if item["displayName"] == name]
    if len(matches) != 1:
        raise RuntimeError(f"Expected exactly one item named {name}")
    return matches[0]


catalog = list_all(f"workspaces/{workspace_id}/items")
ontology = one_named([item for item in catalog if item["type"] == "Ontology"], ONTOLOGY_NAME)
graph = one_named(
    list_all(f"workspaces/{workspace_id}/graphModels"),
    ONTOLOGY_NAME + "_graph_" + ontology["id"].replace("-", ""),
)
lakehouse = one_named([item for item in catalog if item["type"] == "Lakehouse"], "LH_Group_SAP")
properties, _ = api(f"workspaces/{workspace_id}/lakehouses/{lakehouse['id']}")
schema = properties["properties"].get("defaultSchema")
endpoint = notebookutils.conf.get("trident.onelake.endpoint").replace("https://", "").rstrip("/")
table_root = f"abfss://{workspace_id}@{endpoint}/{lakehouse['id']}/Tables/"
table_root += schema + "/" if schema else ""


def gql(query):
    rows = []
    continuation = None
    deadline = time.monotonic() + 180
    while time.monotonic() < deadline:
        path = f"workspaces/{workspace_id}/graphModels/{graph['id']}/executeQuery?beta=true"
        if continuation:
            path += "&continuationToken=" + urllib.parse.quote(continuation, safe="")
        response, _ = api(path, "POST", {"query": query}, "fabriciq-ontology-cli")
        code = response["status"]["code"]
        if code not in ("00000", "02000") or response.get("additionalStatuses"):
            raise RuntimeError(f"Graph query failed or warned: {code}")
        rows.extend(response.get("result", {}).get("data", []))
        continuation = response.get("result", {}).get("nextPage")
        if not continuation:
            if not rows:
                raise RuntimeError("Graph returned no rows; no static fallback is allowed")
            return rows
        time.sleep(2)
    raise TimeoutError("Graph pagination exceeded 180 seconds")


events = gql("""
MATCH (e:RiskEvent)
RETURN e.event_id AS event_id, e.start_date AS event_start_date
ORDER BY e.event_id LIMIT 200
""".strip())
if len(events) >= 200:
    raise RuntimeError("Graph event query reached its limit")
paths = []
for event in events:
    event_id = event["event_id"]
    if not re.fullmatch(r"[A-Z0-9_]{1,40}", event_id):
        raise ValueError("Unexpected event ID")
    rows = gql(f"""
MATCH (e:RiskEvent)-[:EventHasEffect]->(f:FinancialEffect)
      -[:EffectRealizedInMetric]->(m:FinancialMetric),
      (f)-[:EffectOnActivity]->(a:Activity)
WHERE e.event_id='{event_id}' AND m.metric_code='EBITDA' AND m.scenario_tag='actual'
RETURN e.event_id AS event_id, e.start_date AS event_start_date,
       f.effect_id AS effect_id, a.entity_id AS entity_id, m.metric_id AS metric_id
ORDER BY f.effect_id LIMIT 1000
""".strip())
    if len(rows) >= 1000:
        raise RuntimeError("Graph path query reached its limit")
    paths.extend(rows)


def source(table, columns):
    return spark.read.format("delta").load(table_root + table).select(*columns).toPandas()


metrics = source("semantic_financial_metrics_daily",
                 ["metric_id", "metric_date", "entity_id", "metric_code", "scenario_tag", "amount_group"])
effects = source("semantic_event_financial_effects",
                 ["effect_id", "event_id", "entity_id", "effect_date", "delta_ebitda_full_jpy"])
links = source("semantic_effect_metric_links", ["effect_id", "metric_id"])
event_source = source("semantic_risk_events", ["event_id", "start_date"])
graph_events = pd.DataFrame(events)
graph_events["event_start_date"] = pd.to_datetime(graph_events["event_start_date"], utc=True).dt.tz_localize(None).dt.normalize()
event_source["start_date"] = pd.to_datetime(event_source["start_date"]).dt.normalize()
checked = graph_events.merge(event_source, on="event_id", validate="one_to_one")
if len(checked) != len(graph_events) or len(checked) != len(event_source):
    raise ValueError("Graph/Lakehouse event membership differs")
if not (checked["event_start_date"] == checked["start_date"]).all():
    raise ValueError("Graph/Lakehouse event dates differ")
graph_paths = pd.DataFrame(paths)
graph_paths["event_start_date"] = pd.to_datetime(graph_paths["event_start_date"], utc=True).dt.tz_localize(None)
frame, base_columns, audit = prepare_features(metrics, effects, links, graph_paths)
report = run_comparison(frame, base_columns, time_budget=20)
report["source"] = audit
report["graph_event_count"] = len(events)
report["graph_api"] = "Ontology Graph GQL beta"
report["executed_at_utc"] = dt.datetime.now(dt.timezone.utc).isoformat()
report["assumptions"] = [
    "合成データ。為替は過去のECB実績。",
    "Graphの現在のスナップショットを利用。イベント開始日と影響日を利用可能日時と仮定し、厳密な過去時点の公開履歴は未検証。",
    "各予測時点より後のイベント、影響額、為替実績を特徴量に使わない。",
    "四半期の観測は1回、Q4は既に閲覧済み。差は探索的で一般化・因果を示さない。",
    "教師データの92日間は週ごとに重複する。独立した観測数ではない。",
    "予測値の精度保証なし。将来の為替と未知イベントは予測していない。",
]


def render(report):
    lines = []
    for row in report["results"]:
        lines.append(
            "<tr><td>{}</td><td>{:.2f}</td><td>{:.2f}</td><td>{:+.2f}</td>"
            "<td>{:+.2f}%</td><td>{:.2f}</td></tr>".format(
                row["model"], row["q4_forecast_oku"], row["q4_actual_oku"],
                row["q4_error_oku"], row["q4_error_percent"], row["next_q_forecast_oku"],
            )
        )
    model_rows = "".join(
        "<tr><td>{}</td><td>{}</td><td>{:.3f}</td><td>{}</td><td>{}</td></tr>".format(
            item["model"], item["best_estimator"], item["validation"]["validation_mae_company_oku"],
            item["q4_training_rows"], item["future_training_rows"],
        ) for item in report["models"]
    )
    notes = "".join("<li>" + value + "</li>" for value in report["assumptions"])
    return f"""
<div style="font-family:Arial,'Yu Gothic',sans-serif;max-width:1100px;color:#1b2940;line-height:1.55">
<h2>Ontology × AutoML｜次の四半期のEBITDA予測</h2>
<p>実Graphでイベント → 影響額 → 日次指標、および影響額 → 取引活動の関係を取得。
Lakehouseから金額を読み、予測はFLAMLが実施。Graph {report['source']['graph_paths']:,}経路を照合。</p>
<p><b>単位：億円</b>。Q4は2026年7〜9月。次の四半期は2026年10〜12月。
8社の92日予測を合算した全額ベース。</p>
<table style="border-collapse:collapse;width:100%" border="1" cellpadding="8">
<tr><th>方法</th><th>Q4予測</th><th>Q4実績</th><th>差額</th><th>誤差</th><th>次の四半期予測</th></tr>
{''.join(lines)}</table>
<h3>モデル選定と学習行</h3>
<p>2026-03-30を予測起点とする92日先（3Qまで）で選定。評価用Q4と重ならないよう、
その選定時の学習ラベルは起点以前に終了。特徴量あり・なしは同じ検索条件。</p>
<table style="border-collapse:collapse;width:100%" border="1" cellpadding="8">
<tr><th>方法</th><th>選択されたモデル</th><th>選定時の会社別MAE（億円）</th><th>Q4予測学習行</th><th>次Q予測学習行</th></tr>
{model_rows}</table>
<h3>前提と限界</h3><ul>{notes}</ul>
<p>実行UTC: {report['executed_at_utc']}。OntologyとLakehouseは読取専用、結果はこのNotebookに保存。</p>
</div>
"""


html = render(report)
display(HTML(html))


def lro(path, body):
    data, headers = api(path, "POST", body)
    operation = headers.get("x-ms-operation-id")
    if not operation:
        return data
    for _ in range(60):
        time.sleep(2)
        state, _ = api("operations/" + operation)
        if state["status"] == "Succeeded":
            if path.endswith("getDefinition?format=ipynb"):
                return api("operations/" + operation + "/result")[0]
            return state
        if state["status"] in ("Failed", "Cancelled"):
            raise RuntimeError("Notebook persistence failed: " + state["status"])
    raise TimeoutError("Notebook persistence exceeded 120 seconds")


saved = lro(f"workspaces/{workspace_id}/notebooks/{notebook_id}/getDefinition?format=ipynb", {})
part = next(part for part in saved["definition"]["parts"] if part["path"].endswith(".ipynb"))
nb = json.loads(base64.b64decode(part["payload"]))
dashboard = next(cell for cell in nb["cells"] if "forecast-dashboard" in cell["metadata"].get("tags", []))
dashboard["outputs"] = [{
    "output_type": "display_data", "metadata": {},
    "data": {"text/html": [html], "text/plain": ["Ontology × AutoML: saved live result"]},
}]
nb["metadata"]["trading_forecast"] = {"latest_live_report": report}
content = base64.b64encode(json.dumps(nb, ensure_ascii=False).encode()).decode()
lro(f"workspaces/{workspace_id}/notebooks/{notebook_id}/updateDefinition", {
    "definition": {"format": "ipynb", "parts": [{
        "path": "notebook-content.ipynb", "payloadType": "InlineBase64", "payload": content,
    }]}
})
print(json.dumps(report, ensure_ascii=False))
